# ATHLLM — Scientific Rebuild Lab

## From theory → hypothesis → mathematics → architecture → experiment → training → RL → evaluation

This notebook is the **research control room** for rebuilding ATHLLM rather than merely loading or fine-tuning an existing model. Every major architectural choice must have: (1) an intuition, (2) a mathematical statement, (3) an engineering implementation, (4) an ablation, and (5) a measurable success criterion.

We use publicly documented research practices from frontier labs as methodological inspiration—not private internal procedures. Exact OpenAI/Anthropic training recipes, datasets, hardware, and architecture details are often undisclosed. OpenAI explicitly stated that GPT-4's report withheld architecture, hardware, training compute, dataset construction, and detailed training methods for competitive/safety reasons. citeturn249000search47


# 0. The scientific loop

```text
Question
  ↓
Literature + prior evidence
  ↓
Hypothesis
  ↓
Mathematical model
  ↓
Toy experiment
  ↓
Architecture decision
  ↓
Implementation
  ↓
Small-scale training
  ↓
Ablation + scaling study
  ↓
Failure analysis
  ↓
Post-training / RL
  ↓
Evaluation + red team
  ↓
Revision of hypothesis
  ↺
```

This is consistent with the public record: OpenAI has published work emphasizing predictable scaling and empirical scaling laws, while Anthropic publicly describes research spanning alignment, interpretability, red teaming, and agent/context engineering. citeturn513870search0turn249000search1turn249000search6


# 1. Research question

**Primary question:** Can a compact ATHLLM model achieve a better capability/compute ratio by combining a Spark-X2.5-inspired hybrid attention backbone with high-quality teacher distillation, verifiable reasoning/coding RL, and an eventual Token Multi-Access mechanism?

**Null hypothesis H₀:** architectural changes do not improve capability after controlling for parameter count, training tokens, and compute.

**Alternative H₁:** the proposed changes improve one or more target capabilities without unacceptable regressions elsewhere.

We are not allowed to assume H₁ is true before the experiments.


# 2. Define the design constraints

### Capability constraints
- bilingual Arabic/English
- technical + mathematical reasoning
- coding and repository repair
- tool use / agentic behavior
- long-context retrieval and synthesis

### Hardware constraints
- constrained single/limited GPU experiments
- checkpoint continuation across sessions
- memory-aware training
- cheap inference after quantization

### Scientific constraints
- fixed held-out evaluation sets
- contamination control
- reproducible seeds/configs
- equal-compute ablations
- no benchmark claim without a frozen protocol


# 3. Literature map: what we can actually learn publicly

### OpenAI
- Scaling laws: language-model loss was found to follow empirical power-law behavior with model size, data, and compute; this motivates treating scaling as an experimental variable rather than guessing a token count. citeturn513870search0
- GPT-4: the public report says the base model was next-token pretrained and then post-trained with RLHF, while also emphasizing predictable scaling infrastructure; many detailed recipe details were withheld. citeturn249000search0turn249000search47
- o1/o-series: OpenAI publicly reports large-scale RL for reasoning and improvements with both more RL compute and more test-time thinking compute. citeturn513870search8turn249000search9

### Anthropic
- Constitutional AI publicly describes critique/revision plus RL from AI feedback, showing how supervision can be partially generated by models and principles rather than only humans. citeturn513870search1
- Context engineering: Anthropic describes managing system instructions, tools, external data and memory as an engineering problem for long-horizon agents. citeturn513870search2
- Long-running agents and multi-agent research: Anthropic publicly describes context resets, persistent artifacts, lead/subagent loops, and iterative search. citeturn513870search6turn513870search10
- Automated research: Anthropic's 2026 alignment work shows a public example of an LLM autonomously running many candidate experiments and selecting improvements by evaluation. citeturn249000search8


In [ ]:
import torch, yaml, math, json, os, subprocess
from pathlib import Path
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0)
    print('GPU:',p.name,'VRAM GB:',round(p.total_memory/1024**3,2))
cfg=yaml.safe_load(Path('configs/distillation_4b_kaggle.yaml').read_text())


# 4. Start from first principles: language modeling objective

Given tokens `x₁ … x_T`, the autoregressive objective is

**L_NTP = - Σₜ log pθ(xₜ | x_<t)**

and the dataset cross-entropy is

**L = E[-log pθ(xₜ | x_<t)]**.

Perplexity is `PPL = exp(L)` when the loss is measured in nats/token. The first research question is therefore not “what architecture looks modern?” but “what architecture gives a lower loss for a given compute budget and then transfers that capability to downstream tasks?”


# 5. Tokenizer theory

A tokenizer is a compression scheme for text. For a text sequence with `B` bytes and `T` tokens, tokenization efficiency can be tracked as `B/T` bytes per token. A bilingual Arabic/English model should measure this separately for Arabic, English, code, numbers, and mixed-script text.

We must compare tokenizer candidates on: vocabulary utilization, fertility (tokens per word/byte), Arabic morphology, code identifiers, Unicode normalization, and downstream perplexity—not just vocabulary size.


In [ ]:
def token_efficiency(text, tokens):
    return {'bytes':len(text.encode('utf-8')),'tokens':len(tokens),'bytes_per_token':len(text.encode('utf-8'))/max(1,len(tokens))}
print(token_efficiency('مثال عربي mixed English code', list('مثال عربي mixed English code')))


# 6. Transformer mathematics

For hidden states `X ∈ R^(T×d)`:

**Q = XW_Q, K = XW_K, V = XW_V**

**Attention(X) = softmax((QKᵀ / √d_h) + M)V**

where `M` is the causal/local mask.

The residual block is conceptually:

**H' = H + Attention(Norm(H))**

**H'' = H' + MLP(Norm(H'))**

Pre-norm is used here because it makes optimization behavior easier to stabilize in deep residual networks. We will verify this experimentally rather than treating the convention as sacred.


# 7. Why GQA?

With standard multi-head attention, K and V each have `h` heads. With grouped-query attention, Q has `h_q` heads while K/V use `h_kv < h_q` heads.

KV-cache elements per layer are approximately:

**2 × T × h_kv × d_h**

instead of `2 × T × h_q × d_h`.

For 16 Q heads and 4 KV heads, the theoretical K/V cache storage is reduced by a factor of 4 relative to 16/16 MHA at equal head dimension. The cost is that multiple query heads share K/V representations.


In [ ]:
Q_heads,KV_heads,head_dim=16,4,160
print('KV-cache relative fraction vs MHA:',KV_heads/Q_heads)
print('theoretical cache reduction factor:',Q_heads/KV_heads)


# 8. RoPE from geometry

Rotary position embedding can be viewed as rotating pairs of coordinates by a position-dependent angle. For a 2-D pair `(x₁,x₂)`,

**R(θ) [x₁,x₂]ᵀ = [cosθ·x₁ - sinθ·x₂, sinθ·x₁ + cosθ·x₂]ᵀ**.

Applying these rotations to Q and K makes their dot products depend on relative positional phase differences. We will test the actual implementation against a reference numerical implementation before long runs.


# 9. Hybrid attention: the architectural hypothesis

The baseline pattern is

**S S S F | S S S F | …**

where S is sliding-window attention and F is full attention.

For a sequence of length `T` and window `w`, the approximate attention-pair count is:

- full layer: `O(T²)`
- sliding layer: `O(Tw)`

With a 3:1 schedule, most layers use local connectivity while periodic global layers create information paths across distant positions.

### Hypothesis
Periodic global layers may preserve much of the global reasoning/communication ability while reducing average attention cost. This is a hypothesis to test with equal-compute ablations.


In [ ]:
def pair_count(T,w=None): return T*T if w is None else T*min(T,w)
for T in [2048,4096,8192,16384,32768]:
    print(f'T={T:>5} full={pair_count(T)/1e6:>9.1f}M  local512={pair_count(T,512)/1e6:>8.1f}M')


# 10. Architecture reconstruction workflow

We should not jump from a PDF/README directly to a 4B run.

**Pass 1 — symbols:** map every published dimension and tensor.

**Pass 2 — toy model:** implement the equations at tiny dimensions.

**Pass 3 — invariants:** verify tensor shapes, causal masking, parameter counts, checkpoint round-trip, and numerical stability.

**Pass 4 — micro-training:** verify loss decreases on a tiny corpus.

**Pass 5 — ablation:** compare alternative attention schedules under equal parameters and approximately equal compute.

**Pass 6 — scaling:** fit empirical loss/throughput curves before committing to a large run.


In [ ]:
from athllm.models.spark25 import Spark25Config, ATHLLMSpark25
small=Spark25Config(vocab_size=4096,hidden_size=256,intermediate_size=1024,num_hidden_layers=4,num_attention_heads=8,num_key_value_heads=2)
m=ATHLLMSpark25(small)
x=torch.randint(0,small.vocab_size,(2,64))
with torch.no_grad(): y=m(x)
print('input',tuple(x.shape),'logits',tuple(y.shape),'params',sum(p.numel() for p in m.parameters()))


# 11. Parameter-count and compute discipline

A rough dense-transformer training-FLOP estimate is often expressed on the order of `6ND` for `N` parameters and `D` training tokens, with the exact constant depending on architecture and implementation. Treat this as a planning approximation, not a law.

The scientific question is: **given a fixed compute budget, where should we spend it—parameters, tokens, sequence length, or post-training?** OpenAI's scaling-law work directly motivates measuring this rather than guessing. citeturn513870search0turn513870search5


In [ ]:
def rough_dense_flops(params,tokens): return 6*params*tokens
for params,tokens in [(4e9,5e10),(4e9,1e11),(7e9,1e11)]:
    print(params/1e9,'B params ×',tokens/1e9,'B tokens →',rough_dense_flops(params,tokens)/1e18,'EFLOP')


# 12. Data science: build the mixture, not a giant dump

The 5T target is a **capacity target**. It is not scientifically justified to assume “more raw tokens = better model.”

For every source we want a measurable record:

`source → license → language → domain → quality → duplicate rate → contamination risk → token count → training weight`

Keep an immutable manifest version. Evaluation data should be excluded before teacher generation and before training.


# 13. Semantic quality model

Define a document quality score as a weighted vector rather than one opaque number:

`Q = w_p P + w_l L + w_d D + w_s S + w_c C - w_r R`

where `P`=provenance confidence, `L`=language quality, `D`=domain value, `S`=semantic coherence, `C`=code correctness where applicable, and `R`=repetition/spam risk.

The weights must be calibrated on a manually reviewed sample. A heuristic score that has never been validated is only a ranking feature, not a truth label.


# 14. Teacher distillation

For a teacher distribution `p_T` and student `p_S`, temperature-scaled KD can be written as:

**L_KD = T² · KL(p_T^T || p_S^T)**

and supervised loss as `L_SFT = CE(y, p_S)`.

A composite objective may be:

**L = αL_SFT + βL_KD + γL_verified**.

We should only use teacher logits when the checkpoint/API and terms permit it. Otherwise use verified response-level distillation.


# 15. Teacher-data funnel

```text
10M candidate tasks
       ↓ cheap filters
1M hard candidates
       ↓ expensive teacher
teacher trajectories / patches / tool traces
       ↓ independent verification
200K–500K high-value examples
       ↓
ATHLLM-4B SFT + KD
```

This concentrates expensive inference where it has the highest expected information value. Counts are starting budgets, not guaranteed optimal values.


# 16. Reasoning RL

OpenAI publicly reports that o-series reasoning models are trained with large-scale RL on chain-of-thought and that more RL compute and more test-time thinking can improve performance. citeturn513870search8turn249000search9

For ATHLLM, the reward should primarily be **outcome-verifiable**:

`prompt → rollout → verifier → reward`

Examples: exact math answer, executable proof/check, unit-test success, structured constraint satisfaction.

The central research variable is not merely “RL yes/no” but **what reward signal actually produces robust reasoning instead of reward hacking**.


# 17. RL track A — verifiable reasoning

Suggested loop:

```text
sample k responses
      ↓
independent verifier
      ↓
group-relative advantage
      ↓
policy update
      ↓
held-out reasoning eval
```

A GRPO-style group-relative update is a candidate implementation because it can work without training a large separate value model. This is an engineering hypothesis, not a statement about a private frontier-lab recipe.


# 18. RL track B — coding / agent execution

The reward should come from the environment:

`patch applies + tests pass + task requirements satisfied`

rather than from an LLM judge alone.

Anthropic's public agent research emphasizes context, tools, iterative work, and long-horizon execution; therefore coding capability should be evaluated as an agent loop, not just static code generation. citeturn513870search2turn513870search6turn513870search10


# 19. RL track C — self-improvement

```text
model proposes task
      ↓
environment/scaffold
      ↓
model solves task
      ↓
independent verifier
      ↓
successful + failed trajectories
      ↓
failure mining / curriculum
      ↓
next RL round
```

This is the part closest to an automated-research workflow. Anthropic has publicly described 2026 experiments in which an AI research system ran many candidate interventions and selected improvements using evaluation. citeturn249000search8

But self-generated data must be independently verified; otherwise the model can simply reinforce its own mistakes.


# 20. Alignment / behavioral training

Capability training alone is insufficient for an agentic model. Anthropic publicly describes Constitutional AI as critique/revision followed by RL using AI-generated feedback, while OpenAI publicly describes RLHF and later safety training approaches. citeturn513870search1turn249000search0turn249000search12

For ATHLLM, keep a separate behavior/evaluation matrix so that capability gains are not silently obtained by increasing unsafe or unreliable behavior.


# 21. Long-horizon context is a system, not just a number

A 1M context window does not imply useful 1M reasoning. Anthropic's public context-engineering work emphasizes relevance, compaction, structured notes, tools, and persistent memory for long tasks. citeturn513870search2turn513870search6

ATHLLM experiments should therefore separate:
- raw context length;
- retrieval accuracy;
- memory persistence;
- context compaction quality;
- agent task completion over multiple windows.


# 22. Token Multi-Access hypothesis

After the baseline is reproduced, test:

```text
token state
 ├── local attention
 ├── periodic global attention
 ├── compressed memory
 ├── reasoning workspace
 └── tool/environment state
             ↓
       learned router
             ↓
        next hidden state
```

### Prediction
If Token Multi-Access gives better retrieval/reasoning at equal compute, it supports the architectural hypothesis. If not, discard or redesign it. The experiment, not intuition, gets the final vote.


# 23. Ablation matrix

Run controlled experiments:

| Variant | Main question |
|---|---|
| SSSF | Spark-inspired baseline |
| SSSS | what if everything is local? |
| FFFF | what is gained by global attention at small T? |
| SSFS | does global-layer spacing matter? |
| SSSF + long-context curriculum | does training strategy dominate architecture? |
| SSSF + Token Multi-Access | does the new mechanism add capability? |

Hold parameter count and training compute approximately fixed. Report confidence intervals where repeated seeds are affordable.


# 24. Scaling experiments

Train several small/medium points before the expensive model:

`0.1B → 0.3B → 1B → 4B`

For each point measure:
- validation loss;
- tokens/sec;
- peak memory;
- FLOPs estimate;
- reasoning score;
- coding score;
- long-context score.

Fit empirical curves and ask whether the scaling trend is smooth enough to justify the next jump. OpenAI's scaling-law work explicitly frames training as an empirical allocation problem. citeturn513870search0


# 25. Failure analysis loop

Every regression becomes a research artifact:

`failed example → categorize failure → hypothesize cause → design targeted data/architecture change → rerun ablation → accept/reject`

Categories should include: retrieval failure, arithmetic error, planning error, hallucination, tool misuse, context loss, reward hacking, syntax failure, test failure, Arabic morphology/language failure, and alignment failure.


# 26. Frontier evaluation

Never write “better than frontier” before the numbers exist.

For every comparison freeze:
- exact checkpoint/model version
- tokenizer
- system/prompt template
- context window
- tool availability
- decoding parameters
- benchmark version
- evaluator version
- hardware/runtime

Then report absolute results and regressions. A frontier comparison is a measurement protocol, not a marketing statement.


# 27. Reproducibility ledger

Each run should save:

```yaml
git_commit:
config_hash:
dataset_manifest_hash:
tokenizer_hash:
checkpoint:
seed:
gpu:
precision:
sequence_length:
global_batch_size:
gradient_accumulation:
optimizer:
learning_rate:
warmup_tokens:
training_tokens:
eval_suite_version:
```

In [ ]:
commit=subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
print(json.dumps({'git_commit':commit,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},indent=2))


# 28. Kaggle execution strategy

Kaggle should be the **experimental accelerator**, not the place where we pretend to reproduce a frontier-scale pretraining run.

Use Kaggle for:
- toy and micro-model validation;
- short SFT/distillation runs;
- QLoRA experiments;
- verifier-based RL pilots;
- quantization calibration;
- benchmark regression;
- checkpoint continuation.

For large teacher generation or multi-trillion-token training, use external authorized compute/storage and feed the resulting artifacts back through immutable manifests.


# 29. Research ethics / IP boundary

We can learn from public papers, public model cards, released code, and released checkpoints. We should not pretend to know private OpenAI/Anthropic training recipes, and we should not place proprietary weights or restricted datasets into GitHub.

The goal is to reproduce **scientific reasoning and engineering discipline**, not private information.


# 30. Current ATHLLM implementation gap

The repository currently has the architectural reference and planning contracts, but the following are still research engineering work: optimized attention kernels, exact checkpoint conversion validation, tokenizer/data streaming, real distillation loss, production-grade verifiers, actual RL trainers, long-context memory system, automated evaluation, and calibrated INT4 export.

This notebook is the specification against which those implementations should be judged.


# 31. The rule for every future commit

Before adding a major feature, answer five questions in the experiment log:

1. **What phenomenon are we trying to improve?**
2. **What is the mathematical/architectural hypothesis?**
3. **What is the cheapest experiment that could falsify it?**
4. **What baseline/ablation controls the conclusion?**
5. **What result makes us keep, modify, or delete the idea?**

That is the discipline that turns ATHLLM from a model implementation into a research program.


## Public methodology references

- OpenAI — Scaling laws for neural language models: https://openai.com/index/scaling-laws-for-neural-language-models/
- OpenAI — How AI training scales: https://openai.com/index/how-ai-training-scales/
- OpenAI — GPT-4 research / technical report: https://openai.com/index/gpt-4-research/
- OpenAI — Learning to reason with LLMs: https://openai.com/index/learning-to-reason-with-llms/
- Anthropic — Claude's Constitution: https://www.anthropic.com/news/claudes-constitution
- Anthropic — Effective context engineering: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
- Anthropic — Effective harnesses for long-running agents: https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents
- Anthropic — Multi-agent research system: https://www.anthropic.com/engineering/multi-agent-research-system


# 32. ATHLLM as an Arabic AI education + company-building lab

ATHLLM is not only a model repository. It is intended to teach Arabic-speaking learners how modern AI research becomes engineering, products, and companies.

The learning chain is:

**theory → model → experiment → system → customer problem → MVP → pricing → distribution → economics.**


## 33. One technical experiment, two lessons

Example:

**GQA** → fewer KV heads → lower KV-cache storage → potentially lower inference memory → potentially lower serving cost.

The scientific lesson is about attention architecture. The company lesson is that an architectural choice can change unit economics.

Another example:

**better Arabic tokenization** → improved token efficiency on Arabic workloads → potentially lower latency/cost → potentially stronger economics for Arabic-first products.

These are hypotheses that must be measured.


## 34. What the student should build

A learner should progress from:

tokenizer → tiny transformer → attention/GQA experiment → data pipeline → model training → distillation → verifier-based RL → agent environment → deployed AI product → first users → pricing → retention → unit economics.

This makes the repository a practical bridge between AI science, AI engineering, entrepreneurship and product work.


## 35. Arabic-first teaching format

For each difficult concept, provide:

**English technical concept + Arabic explanation + equation + implementation + experiment + business implication.**

Research notebooks should document failures and negative results as well as successful runs. This prevents the educational material from becoming marketing-only content.


## 36. Marketing and credibility

Marketing for ATHLLM should show the work: architecture diagrams, Arabic AI explanations, benchmark methodology, experiment results, cost/performance measurements, tutorials, student projects, product demos and build-in-public reports.

Do not use unverified benchmark numbers or unsupported claims of universal frontier superiority as marketing copy. The strongest educational marketing asset is reproducible evidence.


## 37. Company-building capstone

```text
real Arabic/MENA problem
        ↓
customer interviews
        ↓
ICP + problem definition
        ↓
AI solution + MVP
        ↓
model quality + latency + cost
        ↓
pricing + distribution
        ↓
users + retention + revenue
        ↓
iteration
```

The final lesson is not simply how to train a model. It is how to turn technical capability into measurable customer value.


## 38. Program success metrics

**Model:** loss, reasoning, coding, agent success, Arabic quality, long-context retrieval, latency, memory.

**Engineering:** reproducibility, throughput, utilization, checkpoint reliability, deployment reliability, cost per million tokens.

**Product:** activation, retention, task completion, customer cost, revenue, gross margin.

**Education:** completed labs, reproducible experiments, student projects, contributions, tutorials and launched products.

A model benchmark alone cannot establish success of the overall ATHLLM mission.
